In [16]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("VECTARA_API_KEY")
CORPUS_KEY = "EMPLOYEE"

BASE_URL = "https://api.vectara.io/v2"

### **Guardrails**

In [17]:
BLOCKED_KEYWORDS = [
    "hack",
    "password",
    "bomb",
    "malware",
    "exploit",
    "terrorist"
]


def validate_query(query: str):

    if len(query.strip()) == 0:
        return False, "Query cannot be empty."

    if len(query) > 500:
        return False, "Query too long."

    lower = query.lower()

    for word in BLOCKED_KEYWORDS:
        if word in lower:
            return False, f"Blocked query detected: {word}"

    return True, ""

### **Ingestion**

In [18]:
import requests

def upload_document(file_path):

    url = f"{BASE_URL}/corpora/{CORPUS_KEY}/upload_file"

    headers = {
        "x-api-key": API_KEY
    }

    files = {
        "file": open(file_path, "rb")
    }

    response = requests.post(
        url,
        headers=headers,
        files=files
    )

    if response.status_code == 200:
        print("Document uploaded successfully.")
    else:
        print(response.text)

In [19]:
CONFIDENCE_THRESHOLD = 0.75

def ask_vectara(query):

    url = f"{BASE_URL}/corpora/{CORPUS_KEY}/query"

    payload = {
        "query": query,
        "search": {
            "limit": 5,
            "lexical_interpolation": 0.03,
            "context_configuration": {
                "sentences_before": 2,
                "sentences_after": 2
            },
            "reranker": {
                "type": "customer_reranker",
                "reranker_name": "Rerank_Multilingual_v1"
            }
        },

        "generation": {
            "generation_preset_name":
                "vectara-summary-ext-24-05-med-omni",
            "response_language": "eng",
            "max_used_search_results": 5,
            "enable_factual_consistency_score": True
        }

    }

    headers = {

        "Content-Type": "application/json",
        "Accept": "application/json",
        "x-api-key": API_KEY

    }

    response = requests.post(
        url,
        headers=headers,
        json=payload
    )
    response.raise_for_status()
    return response.json()


def generate_answer(query):
    result = ask_vectara(query)
    factual_score = result.get("factual_consistency_score", 0)
    if factual_score < CONFIDENCE_THRESHOLD:
        return {
            "answer":
                "I couldn't find enough reliable information in the uploaded documents to answer this confidently.",
            "score": factual_score
        }

    return {
        "answer": result["summary"],
        "score": factual_score
    }

In [20]:
print("=" * 60)
print("Vectara Enterprise RAG Demo")
print("=" * 60)

print("\nUploading knowledge base...\n")

# Upload the doc 
upload_document("employee_handbook.pdf")

print("Knowledge Base Ready.")

# Query session
while True:
    query = input("\nAsk Question (type exit): ")
    if query.lower() == "exit":
        break

    valid, message = validate_query(query)

    if not valid:
        print(f"\nGuardrail Blocked: {message}")
        continue

    try:

        result = generate_answer(query)
        print("\nAnswer\n")
        print(result["answer"])
        print("\nHallucination Score")
        print(result["score"])

    except Exception as e:
        print(e)

Vectara Enterprise RAG Demo

Uploading knowledge base...

{"id":"employee_handbook.pdf","metadata":{"access_permission:can_print_faithful":"true","pdf:PDFVersion":"1.4","pdf:hasXFA":"false","access_permission:modify_annotations":"true","pdf:producer":"PyPDF2","access_permission:extract_for_accessibility":"true","access_permission:assemble_document":"true","xmpTPg:NPages":"7","pdf:hasXMP":"false","dc:format":"application/pdf; version=1.4","access_permission:extract_content":"true","access_permission:can_print":"true","access_permission:fill_in_form":"true","pdf:hasCollection":"false","X-TIKA:Parsed-By":"org.apache.tika.parser.pdf.PDFParser","pdf:encrypted":"false","access_permission:can_modify":"true","pdf:docinfo:producer":"PyPDF2","pdf:hasMarkedContent":"false","Content-Type":"application/pdf"},"storage_usage":{"bytes_used":7111,"metadata_bytes_used":5707}}
Knowledge Base Ready.

Answer

You can bring your pet to the office only on designated Pet Fridays. Pets must be vaccinated, supe